In [ ]:
import os
import json 
import time 
from parallel_utils import * 

In [ ]:
limit_prompt ='You can explore up to 5 different environments, ranging from 1 to 5.'

In [ ]:
parameters = '1.5b' 

save_path = 'result path'
path = 'model path'

In [ ]:
num_parallel = 50

In [ ]:
system_prompt = '''You are an expert agent operating in the ALFRED embodied environment.
Given a task, you need to reason first in your mind.
Your reasoning process must be enclosed within <think> </think> tags,
for example: <think> reasoning process here </think>.

After thinking, you may take actions. You can either explore multiple parallel environments with multiple actions or take an action in a specific environment.
At the very beginning, every environment have the same status,but each environment is independent, they do not share state changes after actions are taken.
So, parallel actions are executed simultaneously across different environments. The parallel actions are not carried out sequentially.
You must wrap each action in specific environment tags like <env_i> </env_i> to indicate which environment you are acting in.

To take multiple actions at the same time in different environment, use the <parallel> </parallel> tags and wrap each action within its corresponding <env_i> </env_i> tag, where i refers to the i-th environment:

<parallel>
<env_1> possible action 1 </env_1>
...
<env_i> possible action 2 </env_i>
</parallel>

Your output must follow the rules below.'''


history_prompt = """You have already taken multiple actions in multiple parallel environments. Below are the most recent observaitons and the corresponding actions you took: {action_history}
"""

compressed_prompt_initial = """You are an expert agent operating in the ALFRED Embodied Environment. 
Your task is to: {task_description}
Your current observation is: {current_observation} 
Your admissible actions of the current situation are: {admissible_actions}."""

compressed_prompt_process = """You are an expert agent operating in the ALFRED Embodied Environment. 
Your task is to: {task_description}.
Your initial observation is: {initial_observation}.{history_info}
In your last step, your actions, corresponding observations and admissible actions are:\n{last_history}"""

In [ ]:
import os
from vllm import LLM,SamplingParams
from transformers import AutoTokenizer


os.environ['CUDA_VISIBLE_DEVICES'] = '0,1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'

tokenizer = AutoTokenizer.from_pretrained(path) 
model = LLM(model=path,
            tensor_parallel_size=2,
            gpu_memory_utilization=0.7) 

In [ ]:
game_path = 'https://github.com/LePanda026/Code-for-DPEPO/blob/main/data_pipelines/gamefiles/alfworld/gamefiles_eval.json'

data = read_json(game_path) 

todo_files = [(key,value) for key,value in data.items()]

todo_game_files = [elem[-1] for elem in todo_files] 

env_start_time = time.time()
print('Loading Environments...') 
parallel_env = build_parallel_alfworld_envs(gamefiles=todo_game_files,
                                            # env_num=512,
                                            group_n=1,
                                            resources_per_worker={'num_cpus': 0.1},
                                            num_parallel=num_parallel,
                                            num_copied=0,
                                            is_train=False) 

In [ ]:
def parse_vllm_output(output):
    outputs = [] 
    for response in output:
        outputs.append(response.outputs[0].text)
    
    return outputs 

In [ ]:
TASK_TYPES = {
    "pick": "pick_and_place_simple",
    "look": "look_at_obj_in_light",
    "clean": "pick_clean_then_place_in_recep",
    "heat": "pick_heat_then_place_in_recep",
    "cook": "pick_cool_then_place_in_recep",  # 注意：这里可能是 "pick_cook..."？但按你给的保留
    "pick2": "pick_two_obj_and_place"
} 

In [ ]:
sampling_params = SamplingParams(n=1,max_tokens=2048,temperature=0.)

from tqdm import tqdm 
correct_count = 0

conversations = [] 
for game_file in todo_game_files:
    save_dict = {} 
    # Save the game file name
    save_dict['gamefile'] = game_file 
    # Save the task type
    for task_key, task_name in TASK_TYPES.items():
        if task_name in game_file:
            save_dict['task_type'] = task_name
            break
    
    obs,possible_actions = parallel_env.get_start_info_file(game_file) 

    obv,task = obs.split('\n\nYour task is to: ')
    
    initial_prompt = compressed_prompt_initial.format(task_description=task,
                                                      current_observation=obv,
                                                      admissible_actions=possible_actions)
    start_completion = [
        {'role':'system','content':system_prompt},
        {'role':'user','content':initial_prompt + limit_prompt}
    ]
    
    start_prompt = tokenizer.apply_chat_template(start_completion,
                                                 add_generation_prompt=True,
                                                 tokenize=False) 
    
    save_dict['conversation'] = []
    save_dict['conversation'].extend(start_completion) 

    action_manager = {}
    obs_manager = {}
    success = False
    generation_prompt = start_prompt 
    for i in tqdm(range(50)): 
        turn_dict = {} 
        # Generate and extract the output part
        vllm_response = model.generate([generation_prompt + '<think>'],sampling_params)
        model_response = parse_vllm_output(vllm_response) 
        actions_w_think = extract_think_and_actions(model_response[0]) 
        actions = actions_w_think['actions'] 
        if len(actions) >= num_parallel:
            success = False
            break
        
        # Interact with the parallel environments
        feedback = parallel_env.step_file(game_file,actions) 
        observations = feedback[0]
        scores_list = feedback[1]
        total_obv =  feedback[-1] 
        
        turn_dict['actions'] = actions
        turn_dict['observations'] = observations
        turn_dict['parallel_num'] = len(observations) 
        turn_dict['turn'] = i 
        save_dict['conversation'].append(turn_dict)


        for score in scores_list: 
            if score == 1:
                success = True 

        if success:
            correct_count += 1
            break 
        admissible_commands = [elem['admissible_commands'] for elem in feedback[3]]

        # Store the action and observation 
        store_idx = 0 
        for key,value in actions.items():
            if key not in action_manager:
                action_manager[key] = [value] 
            else:
                action_manager[key].append(value)
            
            if key not in obs_manager:
                obs_manager[key] = [observations[store_idx]]
            else:
                obs_manager[key].append(observations[store_idx])
            store_idx += 1
        
        
        last_action_obv = ''
        idx = 0
        for env_idx,action in actions.items():
            action_cur_env = action
            obs_cur_env = observations[idx] 
            poa = admissible_commands[idx]
            idx += 1 
            last_action_obv += f'In Environment {env_idx}\n'
            last_action_obv += f'Action: {action_cur_env}\n'
            last_action_obv += f'Observation: {obs_cur_env}\n'
            last_action_obv += f'Admissible Actions: {poa}\n'
        
        # prepare the next prompt
        if i == 0: # In the second step, we do not need history infos 
            history_prompt = ''
        
        else:
            history_prompt = ''
            for env_idx in action_manager.keys():
                action_cur_env = action_manager[env_idx] 
                obs_cur_env = obs_manager[env_idx] 
                history_prompt += f'In Environment {env_idx}\n'
                for history_idx,(action,z_obs) in enumerate(zip(action_cur_env,obs_cur_env)):
                    history_prompt += f'Action {history_idx + 1}: {action}\n'
                    history_prompt += f'Observation {history_idx + 1}: {z_obs}\n'
            
            history_prompt = f'\nYou have already taken multiple actions in multiple parallel environments. Below are the most recent observaitons and the corresponding actions you took:\n{history_prompt}\n'

        generation_prompt = compressed_prompt_process.format(task_description=task,
                                                             initial_observation=obv,
                                                             history_info=history_prompt,
                                                             last_history=last_action_obv) 
        
        generation_completion  = [
            {'role':'system','content':system_prompt},
            {'role':'user','content':generation_prompt+ limit_prompt}
        ]
        
        generation_prompt = tokenizer.apply_chat_template(generation_completion,
                                                          add_generation_prompt=True,
                                                          tokenize=False) 
        
    save_dict['turn'] = i 
    save_dict['success'] = success 
    conversations.append(save_dict) 

In [ ]:
f'{save_path}/{name}_{step}.json'

In [ ]:
import json

with open(f'{save_path}/{name}_{step}.json'.replace('baselines','evaluate'),'w') as f:
    json.dump(conversations,f,indent=4) 